# KhadSan AI รอบ 4 : เพิ่มรูปจากชุดรูปที่ทีมรวบรวมเพิ่ม

รอบ 3 พบว่ารูปจากเว็บจริง (Team-collected set) โมเดลยังทายมะนาวเน่ากับทับทิมผิดเยอะ รอบนี้เลยเอารูปชุดนั้นครึ่งหนึ่งมาเทรน

| | รอบ 4 ทำอะไร |
|---|---|
| ข้อมูลเทรน | ชุดรอบ 3 + รูปมะนาว/ทับทิมจากชุดของทีม 52 รูป (กรอบร่างด้วยโมเดล 11 แล้วคัดด้วยตาทีละรูป) |
| ค่าที่ใช้เทรน | เหมือน 10-12 ทุกอย่าง เทรนซ้ำ 3 seed (`13`, `14`, `15`) |
| ข้อสอบเพิ่ม | `team_collected_test` 287 รูป = อีกครึ่งที่ไม่ได้เอาไปเทรน + ผลไม้ชนิดอื่นทั้งหมด |
| ตัวเทียบ | โมเดล 11 (ตัวที่เว็บใช้) สอบทุกชุดเหมือนกัน |

**ก่อนกด Run all**
1. push โฟลเดอร์ `train/` ล่าสุดขึ้น GitHub
2. Runtime > Change runtime type > **T4 GPU**
3. ใน Drive สร้างโฟลเดอร์ `KhadSan_detect_v4` แล้วอัปโหลด 2 ไฟล์ : `KaggleBoxes_v4.zip`, `TeamTest.zip`
4. `ExternalTest.zip` กับ `NewTest.zip` ใช้ของเดิมใน `KhadSan_detect_v3` ได้เลย (ไม่ต้องอัปใหม่)
5. dataset หลัก : ตั้ง Colab Secret `ROBOFLOW_API_KEY` **หรือ** มี `FruitsQuality.zip` ใน `KhadSan_detect_v4` หรือ `v3`

ใช้เวลาราว 3-4 ชั่วโมง ถ้าหลุดกลางทาง กด Run all ใหม่ การทดลองที่เสร็จแล้วจะถูกข้าม

## 1) เช็กว่าได้ GPU
ถ้าขึ้นว่าไม่เจอ GPU : Runtime > Change runtime type > **T4 GPU** > Save แล้วกด Run all ใหม่

In [ ]:
import torch
if not torch.cuda.is_available():
    raise Exception("ไม่เจอ GPU ไปที่ Runtime > Change runtime type > เลือก T4 GPU แล้วกด Run all ใหม่")
print("ได้ GPU:", torch.cuda.get_device_name(0))

## 2) ติดตั้งไลบรารี
ล็อกรุ่น ultralytics ให้ตรงกับที่ทีมใช้ (8.4.191) ผลจะได้เทียบกันได้

In [ ]:
!pip install -q ultralytics==8.4.191 roboflow
!yolo settings sync=False
import ultralytics
print("ultralytics", ultralytics.__version__)
if ultralytics.__version__ != "8.4.191":
    raise Exception("รุ่น ultralytics ไม่ตรง ให้ Runtime > Restart session แล้วกด Run all ใหม่")

## 3) ดึงโค้ดเทรนจาก GitHub ของทีม
ต้อง **push โค้ดล่าสุดขึ้น GitHub ก่อน** ช่องนี้จะเช็กให้ว่าโค้ดที่ดึงมาเป็นรุ่นที่ใช้กับรอบนี้ได้

In [ ]:
import os
REPO = "https://github.com/srirat-intalai/khadsan-ai"

# ดึงครั้งแรก = clone / รันซ้ำ = อัปเดตเป็นโค้ดล่าสุด
if os.path.isdir("/content/khadsan-ai"):
    !git -C /content/khadsan-ai pull -q
else:
    !git clone -q {REPO} /content/khadsan-ai

CODE = "/content/khadsan-ai/train"
!git -C /content/khadsan-ai log -1 --format="โค้ดรุ่นล่าสุด: %h %s (%ad)" --date=short

# เช็กว่าโค้ดมีของที่รอบนี้ต้องใช้ (ถ้าไม่มี แปลว่ายังไม่ได้ push โค้ดล่าสุด)
train_code = open(CODE + "/train_yolo.py", encoding="utf-8").read()
for need in ["13_team_data", "15_team_data_seed3", "--team-test", "seed_summary_round4"]:
    if need not in train_code:
        raise Exception("train_yolo.py ใน GitHub ยังเป็นรุ่นเก่า (ไม่มี " + need + ") ให้ push โค้ดล่าสุดก่อน")
print("โค้ดครบ พร้อมเทรน")

## 4) ต่อ Google Drive แล้วโหลดข้อมูล
ผลเก็บใน Drive โฟลเดอร์ `KhadSan_detect_v4` (ไม่ทับรอบก่อน)

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

OUT = "/content/drive/MyDrive/KhadSan_detect_v4"
OLD_OUT = "/content/drive/MyDrive/KhadSan_detect_v3"   # zip ที่ใช้ซ้ำจากรอบ 3
os.makedirs(OUT, exist_ok=True)
print("ผลจะเก็บที่", OUT)

# ----- dataset หลัก (Fruits Quality) : ใช้ zip ใน Drive ถ้ามี ไม่มีก็โหลดจาก Roboflow ด้วย API key -----
SRC = "/content/FruitsQuality"
FQ_ZIP = OUT + "/FruitsQuality.zip"
if not os.path.exists(FQ_ZIP):
    FQ_ZIP = OLD_OUT + "/FruitsQuality.zip"
if os.path.exists(FQ_ZIP):
    print("ใช้ FruitsQuality.zip จาก Drive")
    !unzip -q -o "{FQ_ZIP}" -d /content/zip_tmp
    for root, dirs, files in os.walk("/content/zip_tmp"):
        if "data.yaml" in files:
            SRC = root
            break
else:
    from google.colab import userdata
    from roboflow import Roboflow
    rf = Roboflow(api_key=userdata.get("ROBOFLOW_API_KEY"))
    project = rf.workspace("alex-day-qzfwa").project("fruits-quality-qtdtf")
    project.version(1).download("yolov8", location=SRC)

if not os.path.exists(SRC + "/data.yaml"):
    raise Exception("ไม่เจอ data.yaml ใน dataset หลัก เช็กไฟล์ zip หรือ API key")
count = 0
for part in ["train", "valid", "test"]:
    if os.path.isdir(SRC + "/" + part + "/images"):
        count = count + len(os.listdir(SRC + "/" + part + "/images"))
print("dataset หลัก", count, "รูป (ควรได้ 2583)")
if count != 2583:
    print("!!! จำนวนรูปไม่ตรงกับที่ทีมใช้ ผลอาจต่างจากเดิม")


# แตก zip แล้วหาโฟลเดอร์ที่ต้องการ (หาใน v4 ก่อน ไม่เจอค่อยหาใน v3)
def unzip_find(zip_name, folder_name):
    zip_file = OUT + "/" + zip_name
    if not os.path.exists(zip_file):
        zip_file = OLD_OUT + "/" + zip_name
    if not os.path.exists(zip_file):
        raise Exception("ไม่เจอ " + zip_name + " ใน " + OUT + " หรือ " + OLD_OUT)
    tmp = "/content/tmp_" + zip_name.replace(".zip", "")
    !unzip -q -o "{zip_file}" -d "{tmp}"
    for root, dirs, files in os.walk(tmp):
        if os.path.basename(root) == folder_name:
            return root
    raise Exception("ใน " + zip_name + " ไม่มีโฟลเดอร์ " + folder_name)


def check_pairs(folder, should_be):
    images = len(os.listdir(folder + "/images"))
    labels = len(os.listdir(folder + "/labels"))
    print(os.path.basename(folder), images, "รูป (ควรได้", str(should_be) + ")")
    if images != labels:
        raise Exception("รูปกับไฟล์ป้ายไม่เท่ากันใน " + folder)


# ----- ชุดทดสอบภายนอก (Kaggle 600 รูป ไม่มีกรอบ) -----
EXTERNAL = unzip_find("ExternalTest.zip", "external_test")
total = 0
for c in sorted(os.listdir(EXTERNAL)):
    total = total + len(os.listdir(EXTERNAL + "/" + c))
print("external_test", total, "รูป (ควรได้ 600)")

# ----- รูปเทรนเพิ่ม (ชุดรอบ 3 + รูปมะนาว/ทับทิมของทีม 52 รูป) -----
EXTRA = unzip_find("KaggleBoxes_v4.zip", "kaggle_boxes_v4")
check_pairs(EXTRA, 3352)

# ----- ข้อสอบชุดรูปที่ทีมรวบรวมเพิ่ม (ไม่มีกรอบ แยกโฟลเดอร์ตามคลาส) -----
TEAM_TEST = unzip_find("TeamTest.zip", "team_collected_test")
total = 0
for c in sorted(os.listdir(TEAM_TEST)):
    total = total + len(os.listdir(TEAM_TEST + "/" + c))
print("team_collected_test", total, "รูป (ควรได้ 287)")

# ----- ข้อสอบใหม่ 2 ชุด -----
NEW_TEST = unzip_find("NewTest.zip", "new_test")
SAME_SOURCE = unzip_find("NewTest.zip", "same_source_test")
check_pairs(NEW_TEST, 150)
check_pairs(SAME_SOURCE, 370)

## 5) เตรียมข้อมูล
แบ่งชุดเหมือนรอบ 1-2 (seed 42 ได้ชุดเดิมทุกครั้ง) แล้วทำ `data_more.yaml` = ชุดเทรนเดิม + รูปเทรนเพิ่ม (val และ test ใช้ชุดเดิม)

In [ ]:
!python "{CODE}/prepare_data.py" --src "{SRC}" --out /content/data_5fruits --extra "{EXTRA}"

for f in ["data.yaml", "data_more.yaml"]:
    if not os.path.exists("/content/data_5fruits/" + f):
        raise Exception("ไม่มี " + f + " เตรียมข้อมูลไม่สำเร็จ ดูข้อความ error ด้านบน")
for part in ["train", "val", "test"]:
    print(part, len(os.listdir("/content/data_5fruits/" + part + "/images")), "รูป (ควรได้ 1624 / 348 / 348)")
!cp /content/data_5fruits/dataset_counts.csv "{OUT}/"

## 6) เริ่มเทรน 3 การทดลอง แล้วสอบทุกชุด
ลำดับ : `13_team_data` (seed 42) → `14_team_data_seed2` → `15_team_data_seed3`
จากนั้นให้โมเดลที่เพิ่งเทรน + โมเดล 11 (เว็บ) สอบ test เดิม, Kaggle, ข้อสอบใหม่ 2 ชุด และ `team_collected_test`

In [ ]:
# โมเดล 11 = ตัวที่เว็บใช้อยู่ (อยู่ใน GitHub) ไว้เทียบ
COMPARE = "11_new_data_seed2=/content/khadsan-ai/web/models/best_detect.pt"

!python "{CODE}/train_yolo.py" --data /content/data_5fruits/data.yaml --more-data /content/data_5fruits/data_more.yaml --only 13_team_data,14_team_data_seed2,15_team_data_seed3 --out "{OUT}" --external "{EXTERNAL}" --new-test "{NEW_TEST}" --same-source-test "{SAME_SOURCE}" --team-test "{TEAM_TEST}" --compare "{COMPARE}"

## 7) ดูผล

In [ ]:
import glob
import csv
from IPython.display import Image, Markdown, display

print("=== test เดิม (348 รูป) ===")
display(Markdown(open(OUT + "/results/summary.md", encoding="utf-8").read()))
print(open(OUT + "/models/class_info.json", encoding="utf-8").read())

print("=== 3 seed ผลแกว่งแค่ไหน ===")
if os.path.exists(OUT + "/results/seed_summary_round4.md"):
    display(Markdown(open(OUT + "/results/seed_summary_round4.md", encoding="utf-8").read()))

print("=== ชุดรูปที่ทีมรวบรวมเพิ่ม (accuracy = ทายถูกทั้งชนิดและสด/เน่า) ===")
print(open(OUT + "/results/team_tests.csv", encoding="utf-8").read())
for f in sorted(glob.glob(OUT + "/results/team_test_*_accuracy.csv")):
    print(os.path.basename(f))
    print(open(f, encoding="utf-8").read())

print("=== ข้อสอบใหม่ (rotten_called_fresh = ลูกเน่าที่โมเดลบอกว่าสด ยิ่งน้อยยิ่งดี) ===")
print(open(OUT + "/results/new_tests.csv", encoding="utf-8").read())

print("=== ชุดแหล่งอื่น (Kaggle) ===")
for f in sorted(glob.glob(OUT + "/results/external_*_summary.txt")):
    print(open(f, encoding="utf-8").read())

for name in ["compare_seeds_round4", "confusion_matrix_normalized_best"]:
    file = OUT + "/results/" + name + ".png"
    if os.path.exists(file):
        print(name)
        display(Image(file))
print(open(OUT + "/results/per_class_best.csv", encoding="utf-8").read())

## 8) เช็กว่ายังตีกรอบหน้าคนเป็นผลไม้ไหม
ใช้รูปคน (zidane.jpg) กับรูปรถบัส (bus.jpg) ที่มากับ ultralytics ไม่มีผลไม้ในรูป ค่าที่ดีคือไม่เจอกรอบเลย
(ความมั่นใจขั้นต่ำ 0.40 เท่ากับที่เว็บใช้)

In [ ]:
from ultralytics.utils import ASSETS

for name in ["13_team_data", "14_team_data_seed2", "15_team_data_seed3"]:
    best = OUT + "/runs/" + name + "/weights/best.pt"
    if not os.path.exists(best):
        continue
    model = ultralytics.YOLO(best)
    for picture in ["zidane.jpg", "bus.jpg"]:
        r = model.predict(str(ASSETS / picture), conf=0.40, verbose=False)[0]
        found = []
        for i in range(len(r.boxes)):
            found.append(r.names[int(r.boxes.cls[i])] + " " + str(round(float(r.boxes.conf[i]) * 100)) + "%")
        if len(found) == 0:
            print(name, "|", picture, "-> ไม่เจอผลไม้ (ถูกต้อง)")
        else:
            print(name, "|", picture, "-> ตีกรอบผิด:", found)

## 9) เอาไปใช้ต่อ
- ส่งลิงก์โน้ตบุ๊กนี้ (ที่รันเสร็จแล้ว) ให้ทีมตรวจก่อน **ยังไม่ต้องเปลี่ยนโมเดลในเว็บ**
- ใช้ตัวใหม่แทนโมเดล 11 เมื่อครบ 4 ข้อ (เทียบกับแถว `11_new_data_seed2`)
  1. `team_collected_test` ทายถูกมากขึ้น โดยเฉพาะมะนาวเน่ากับทับทิม (`team_test_*_accuracy.csv`)
  2. Test F1 (test เดิม), Kaggle และ `new_test` ไม่ตกมาก
  3. `rotten_called_fresh` ไม่เพิ่ม
  4. ไม่ตีกรอบรูปคน/รถบัส (ข้อ 8)
- ถ้าใช้ตัวใหม่ : `models/best_detect.pt` + `class_info.json` → วางทับใน `web/models/`
- `KhadSan_detect_v4/results/` → วางใน `results/round4/` ของโปรเจกต์ (ไม่ต้องโหลด `runs/`)